# Sequential Ronit IDK Cascade (REAL TIME CLASSIFICATION)

This notebook trains the Random Forest router from saved logits, then runs the skip cascade sequentially with ResNet-18, ResNet-34, and ResNet-50.

This cell imports the packages and finds the repo paths used by the notebook.

In [1]:
from collections import Counter
from pathlib import Path
import tarfile
import time

import numpy as np
import torch
from PIL import Image
from sklearn.ensemble import RandomForestClassifier
from torchvision import models

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

IMAGENETV2_DIR = PROJECT_ROOT / "ImageNet-V2 DataSet"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

This cell sets the models, router training caches, test dataset, and confidence thresholds.

In [2]:
MODEL_A = "resnet18"
MODEL_B = "resnet34"
MODEL_C = "resnet50"
MODELS = (MODEL_A, MODEL_B, MODEL_C)

VARIANT_ARCHIVES = {
    "matched-frequency": IMAGENETV2_DIR / "imagenetv2-matched-frequency.tar.gz",
    "threshold-0.7": IMAGENETV2_DIR / "imagenetv2-threshold0.7.tar.gz",
    "top-images": IMAGENETV2_DIR / "imagenetv2-top-images.tar.gz",
}

TRAIN_CACHE_PREFIXES = ("matched", "top")
TEST_VARIANT = "threshold-0.7"
TEST_SAMPLES = 10000

CLASSIFICATION_THRESHOLD = 0.9
THRESHOLD_SKIP = 0.3
SKIP = 0
PREDICT = 1

if not torch.backends.mps.is_available():
    raise RuntimeError("MPS is required for this sequential notebook")

DEVICE = torch.device("mps")
DEVICE_BY_MODEL = {model_name: str(DEVICE) for model_name in MODELS}

This cell streams images and labels directly from the local ImageNet-V2 tar files.

In [3]:
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")


def label_from_key(key):
    return int(key.split("/")[1])


def stream_rows(variant, max_samples):
    emitted = 0
    with tarfile.open(VARIANT_ARCHIVES[variant], "r:*") as tar:
        for member in tar:
            if not member.isfile() or not member.name.lower().endswith(IMAGE_EXTENSIONS):
                continue

            image_file = tar.extractfile(member)
            image = Image.open(image_file).convert("RGB")
            image_file.close()

            yield image, label_from_key(member.name)
            emitted += 1
            if emitted >= max_samples:
                break

This cell loads the three pretrained ResNet models on one MPS device.

In [4]:
def load_model(model_name):
    specs = {
        "resnet18": (models.resnet18, models.ResNet18_Weights.DEFAULT),
        "resnet34": (models.resnet34, models.ResNet34_Weights.DEFAULT),
        "resnet50": (models.resnet50, models.ResNet50_Weights.DEFAULT),
    }
    factory, weights = specs[model_name]
    model = factory(weights=weights).to(DEVICE).eval()
    return model, weights.transforms()


models_by_name = {}
transforms_by_name = {}

for model_name in MODELS:
    model, transform = load_model(model_name)
    models_by_name[model_name] = model
    transforms_by_name[model_name] = transform

print("Loaded models:", MODELS)
print("Device:", DEVICE)

Loaded models: ('resnet18', 'resnet34', 'resnet50')
Device: mps


This cell defines prediction and Random Forest feature helpers.

In [5]:
def sync_mps():
    torch.mps.synchronize()


def predict(model_name, image):
    image_tensor = transforms_by_name[model_name](image).unsqueeze(0).to(DEVICE)
    sync_mps()
    start = time.perf_counter()

    with torch.inference_mode():
        probabilities = torch.softmax(models_by_name[model_name](image_tensor), dim=1)
        sync_mps()

    elapsed_ms = (time.perf_counter() - start) * 1000.0
    probabilities = probabilities[0].detach().cpu().numpy()
    return probabilities, int(probabilities.argmax()), float(probabilities.max()), elapsed_ms


def probability_features(probabilities):
    probabilities = np.asarray(probabilities)
    confidence = probabilities.max(axis=1)
    entropy = -(probabilities * np.log(probabilities + 1e-12)).sum(axis=1)
    top_two = np.partition(probabilities, -2, axis=1)[:, -2:]
    margin = top_two.max(axis=1) - top_two.min(axis=1)
    return np.column_stack([confidence, entropy, margin]).astype(np.float32)

This cell builds Random Forest training rows from the saved ResNet-18 and ResNet-34 artifact caches.

In [6]:
def load_cache(prefix, model_name):
    with np.load(ARTIFACTS_DIR / f"{prefix}_{model_name}.npz") as data:
        return {"probabilities": data["probabilities"]}


def build_training_data():
    feature_parts = []
    label_parts = []

    for prefix in TRAIN_CACHE_PREFIXES:
        a_cache = load_cache(prefix, MODEL_A)
        b_cache = load_cache(prefix, MODEL_B)
        a_probs = a_cache["probabilities"]
        a_idk = a_probs.max(axis=1) < CLASSIFICATION_THRESHOLD

        X = probability_features(a_probs[a_idk])
        y = np.where(
            b_cache["probabilities"].max(axis=1)[a_idk] < CLASSIFICATION_THRESHOLD,
            SKIP,
            PREDICT,
        ).astype(np.int64)

        feature_parts.append(X)
        label_parts.append(y)
        print(prefix, "training rows:", len(y))

    X = np.concatenate(feature_parts)
    y = np.concatenate(label_parts)
    print("Training rows:", len(y), "skip:", int((y == SKIP).sum()), "predict:", int((y == PREDICT).sum()))
    return X, y


train_X, train_y = build_training_data()

matched training rows: 6930
top training rows: 5667
Training rows: 12597 skip: 10286 predict: 2311


This cell trains the Random Forest router with the paper hyperparameters.

In [7]:
router_name = "random_forest"
router = RandomForestClassifier(
    n_estimators=50,
    max_depth=4,
    min_samples_leaf=40,
    class_weight="balanced",
    random_state=42,
    n_jobs=1,
)
router.fit(train_X, train_y)
print("Router:", router_name)

Router: random_forest


This cell runs the sequential cascade on the test split and stores the metrics.

In [8]:
def run_sequential_cascade():
    labels = []
    final_predictions = []
    latencies_ms = []
    final_count_by_model = Counter()
    execution_count_by_model = Counter()
    execution_time_ms_by_model = Counter()
    heavy_route_count_by_model = Counter()
    router_call_count = 0
    router_time_ms_total = 0.0
    run_start = time.perf_counter()

    for image, label in stream_rows(TEST_VARIANT, TEST_SAMPLES):
        sample_start = time.perf_counter()
        labels.append(int(label))

        a_probs, a_pred, a_conf, a_ms = predict(MODEL_A, image)
        execution_count_by_model[MODEL_A] += 1
        execution_time_ms_by_model[MODEL_A] += a_ms

        if a_conf >= CLASSIFICATION_THRESHOLD:
            final_model = MODEL_A
            final_prediction = a_pred
        else:
            router_start = time.perf_counter()
            route = int(router.predict(probability_features(a_probs[None, :]))[0])
            router_time_ms_total += (time.perf_counter() - router_start) * 1000.0
            router_call_count += 1

            if route == SKIP:
                heavy_route_count_by_model[MODEL_C] += 1
                _, c_pred, _, c_ms = predict(MODEL_C, image)
                execution_count_by_model[MODEL_C] += 1
                execution_time_ms_by_model[MODEL_C] += c_ms
                final_model = MODEL_C
                final_prediction = c_pred
            else:
                heavy_route_count_by_model[MODEL_B] += 1
                _, b_pred, b_conf, b_ms = predict(MODEL_B, image)
                execution_count_by_model[MODEL_B] += 1
                execution_time_ms_by_model[MODEL_B] += b_ms

                if b_conf >= CLASSIFICATION_THRESHOLD:
                    final_model = MODEL_B
                    final_prediction = b_pred
                else:
                    _, c_pred, _, c_ms = predict(MODEL_C, image)
                    execution_count_by_model[MODEL_C] += 1
                    execution_time_ms_by_model[MODEL_C] += c_ms
                    final_model = MODEL_C
                    final_prediction = c_pred

        final_predictions.append(final_prediction)
        final_count_by_model[final_model] += 1
        latencies_ms.append((time.perf_counter() - sample_start) * 1000.0)

    labels = np.asarray(labels, dtype=np.int64)
    final_predictions = np.asarray(final_predictions, dtype=np.int64)
    latencies_ms = np.asarray(latencies_ms, dtype=np.float64)
    total_wall_time_seconds = time.perf_counter() - run_start
    correct_predictions = int(np.count_nonzero(final_predictions == labels))

    return {
        "total_samples": int(len(labels)),
        "accuracy": float(correct_predictions / len(labels)),
        "correct_predictions": correct_predictions,
        "total_wall_time_seconds": float(total_wall_time_seconds),
        "throughput_fps": float(len(labels) / total_wall_time_seconds),
        "mean_latency_ms": float(latencies_ms.mean()),
        "router_name": router_name,
        "router_call_count": int(router_call_count),
        "total_router_time_ms": float(router_time_ms_total),
        "mean_router_time_ms": float(router_time_ms_total / router_call_count) if router_call_count else 0.0,
        "device_by_model": DEVICE_BY_MODEL,
        "final_prediction_count_by_model": {model: int(final_count_by_model[model]) for model in MODELS},
        "execution_count_by_model": {model: int(execution_count_by_model[model]) for model in MODELS},
        "mean_execution_time_ms_by_model": {
            model: float(execution_time_ms_by_model[model] / execution_count_by_model[model])
            if execution_count_by_model[model]
            else 0.0
            for model in MODELS
        },
        "heavy_route_count_by_model": {model: int(heavy_route_count_by_model[model]) for model in (MODEL_B, MODEL_C)},
    }


results = run_sequential_cascade()

This cell prints the relevant metrics for the sequential run.

In [9]:
print("Real-Time Sequential MPS Test")
print("Confidence Threshold Constant: ", CLASSIFICATION_THRESHOLD)
print("Device by model:", results["device_by_model"])
print("Total samples:", results["total_samples"])
print("Accuracy:", round(results["accuracy"], 4))
print("Correct predictions:", results["correct_predictions"])
print("Total wall time (seconds):", round(results["total_wall_time_seconds"], 3))
print("Throughput (FPS):", round(results["throughput_fps"], 3))
print("Mean latency (ms):", round(results["mean_latency_ms"], 3))
print()
print(f"Classifier {results['router_name']} router:")
print(f"  Router calls: {results['router_call_count']}")
print(f"  Total router time (ms): {results['total_router_time_ms']:.3f}")
print(f"  Mean router time (ms): {results['mean_router_time_ms']:.6f}")
print()
print("Final prediction count by model:")
for model_name in MODELS:
    print(f"  {model_name}: {results['final_prediction_count_by_model'][model_name]}")
print("Execution count by model:")
for model_name in MODELS:
    print(f"  {model_name}: {results['execution_count_by_model'][model_name]}")
print("Mean execution time by model (ms):")
for model_name in MODELS:
    print(f"  {model_name}: {results['mean_execution_time_ms_by_model'][model_name]:.3f}")
print("Heavy route count by MPS model:")
for model_name in (MODEL_B, MODEL_C):
    print(f"  {model_name}: {results['heavy_route_count_by_model'][model_name]}")

Real-Time Sequential MPS Test
Confidence Threshold Constant:  0.9
Device by model: {'resnet18': 'mps', 'resnet34': 'mps', 'resnet50': 'mps'}
Total samples: 10000
Accuracy: 0.7749
Correct predictions: 7749
Total wall time (seconds): 386.533
Throughput (FPS): 25.871
Mean latency (ms): 36.558

Classifier random_forest router:
  Router calls: 6278
  Total router time (ms): 17502.178
  Mean router time (ms): 2.787859

Final prediction count by model:
  resnet18: 3722
  resnet34: 807
  resnet50: 5471
Execution count by model:
  resnet18: 10000
  resnet34: 2398
  resnet50: 5471
Mean execution time by model (ms):
  resnet18: 10.683
  resnet34: 16.547
  resnet50: 27.898
Heavy route count by MPS model:
  resnet34: 2398
  resnet50: 3880
